<a href="https://colab.research.google.com/github/eschain93/ban5600_800/blob/main/week7/Homework_7_Chainani_Emma.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Weekly Content Publishing Dashboard

This Python automation was created to solve a recurring business reporting problem: building a weekly publishing pace dashboard from multiple Google Sheets. The manual process required checking several source trackers, reviewing article URLs, confirming publishing status, checking publish dates, grouping the data by brand and topic cluster, and then creating dashboard visuals. In the earlier project blueprint, the goal was to create a dashboard showing month-to-date publishing progress, including brand scorecards, article share by brand, topic cluster breakdowns, and weekly publishing pace.

The completed automation uses Python to pull the source data, clean and validate it, filter it to the current reporting month, summarize the results, and export an interactive HTML dashboard. The final dashboard includes filters, scorecards, charts, a reporting month label, and an AI-generated executive summary.

##Step 1: Authenticate and Connect to Google Sheets

In [ ]:
# ==========================================
# 1. AUTHENTICATE AND CONNECT TO GOOGLE SHEETS
# ==========================================

from google.colab import auth
auth.authenticate_user()

import gspread
import pandas as pd
from google.auth import default

creds, _ = default()
gc = gspread.authorize(creds)

print("Google authentication successful.")
print("Google Sheets connection successful.")

Google authentication successful.
Google Sheets connection successful.


The first section of code connects the notebook to Google Sheets. The code starts by importing Google Colab’s authentication tool and running auth.authenticate_user(). This opens the Google login process so the notebook has permission to access the Google Sheets used in the project.

The code then imports gspread, pandas, and default from google.auth. The gspread library allows Python to open and read Google Sheets. The pandas library is used for working with data in DataFrames, which are table-like structures similar to spreadsheets. The default() function retrieves the credentials created during authentication.

The line gc = gspread.authorize(creds) creates an authorized Google Sheets client. This client is what allows Python to open each spreadsheet by its Sheet ID, access the correct tab, and pull the data into the notebook.

The print statements at the end confirm that Google authentication and the Google Sheets connection were successful. This is useful because it verifies that the setup worked before the automation moves into the data processing steps.

##Step 2: Define the Source Sheets

In [ ]:
# ==========================================
# 2. DEFINE SOURCE SHEETS
# ==========================================

source_sheets = [
    {
        "sheet_id": "1mTjFjYu1ahm9pmY-KrOPVod69Pf0ONJFHg-PZTRRgX0",
        "tab_name": "Working List"
    },
    {
        "sheet_id": "1p8YUfu8aO4qpBHZBnf-Xu06QAL0ehr2kK9KljEd60rc",
        "tab_name": "Topics"
    },
    {
        "sheet_id": "1PTRiDG7-UCVvIy4UIynCTcD9PXUbrC8ZvUYiWDvovTU",
        "tab_name": "Current Koray Content Workstream"
    }
]

print(f"{len(source_sheets)} source sheets have been defined.")

3 source sheets have been defined.


The next section defines the Google Sheets and tabs used as source data. The code creates a list called source_sheets. Each item in the list is a dictionary containing a sheet_id and a tab_name.

This structure is useful because the automation can loop through the list instead of repeating nearly identical code for each source. In this project, the source tabs include:

Working List
Topics
Current Koray Content Workstream

The automation is designed to treat each of these as an input source. Instead of manually opening each sheet, Python reads the list and processes each source one at a time.

This approach also makes the automation easier to update. If a new tracker needs to be added later, another dictionary can be added to the source_sheets list without rewriting the main logic.

##Step 3: Create a Column Audit

In [ ]:
# ==========================================
# 3. CREATE A COLUMN AUDIT
# ==========================================

column_audit = []

for source in source_sheets:
    sheet_id = source["sheet_id"]
    tab_name = source["tab_name"]

    spreadsheet = gc.open_by_key(sheet_id)
    worksheet = spreadsheet.worksheet(tab_name)

    values = worksheet.get_all_values()

    if len(values) == 0:
        print(f"{tab_name} is empty.")
        continue

    header_row = values[0]
    full_sheet_name = spreadsheet.title

    for position, column_name in enumerate(header_row, start=1):
        column_audit.append({
            "full_sheet_name": full_sheet_name,
            "tab_name": tab_name,
            "column_position": position,
            "column_name": column_name
        })

column_audit_df = pd.DataFrame(column_audit)

display(column_audit_df)

,full_sheet_name,tab_name,column_position,column_name
0,[Homework Version] Article Reviewer/Fact Check...,Working List,1,article_url
1,[Homework Version] Article Reviewer/Fact Check...,Working List,2,Wordpress Link
2,[Homework Version] Article Reviewer/Fact Check...,Working List,3,Title
3,[Homework Version] Article Reviewer/Fact Check...,Working List,4,topic_cluster
4,[Homework Version] Article Reviewer/Fact Check...,Working List,5,Assignee
...,...,...,...,...
84,[Homework Version] Xometry SEO Article Trackin...,Current Koray Content Workstream,23,Days Since Writing Publish
85,[Homework Version] Xometry SEO Article Trackin...,Current Koray Content Workstream,24,Refresh Status
86,[Homework Version] Xometry SEO Article Trackin...,Current Koray Content Workstream,25,Main Koray Tracking Sheet Name
87,[Homework Version] Xometry SEO Article Trackin...,Current Koray Content Workstream,26,Jira Sprint


Before cleaning or filtering the data, the automation first inspects the structure of each source sheet. This is important because the original source trackers did not all have the same layouts or column positions.

The code creates an empty list called column_audit. It then loops through each source sheet, opens the spreadsheet by key, selects the worksheet tab, and pulls all values from the tab with worksheet.get_all_values().

The first row is treated as the header row. The code loops through every column name in that header row and records the full sheet name, tab name, column position, and column name. These records are stored in the column_audit list and then converted into a pandas DataFrame called column_audit_df.

This step acts as a data discovery process. It allows the user to see exactly what columns Python found before any filtering is performed. This is important because an automation should not blindly assume the source data is structured correctly.

##Step 4: Infer Brand From the Full Sheet Name

In [ ]:
# ==========================================
# 4. INFER BRAND FROM THE FULL SHEET NAME
# ==========================================

def infer_brand(full_sheet_name):
    sheet_name_lower = full_sheet_name.lower()

    if "thomas" in sheet_name_lower:
        return "Thomas"
    elif "xometry" in sheet_name_lower:
        return "Xometry"
    else:
        return "Unknown"

column_audit_df["inferred_brand"] = column_audit_df["full_sheet_name"].apply(infer_brand)

display(column_audit_df)

,full_sheet_name,tab_name,column_position,column_name,inferred_brand
0,[Homework Version] Article Reviewer/Fact Check...,Working List,1,article_url,Thomas
1,[Homework Version] Article Reviewer/Fact Check...,Working List,2,Wordpress Link,Thomas
2,[Homework Version] Article Reviewer/Fact Check...,Working List,3,Title,Thomas
3,[Homework Version] Article Reviewer/Fact Check...,Working List,4,topic_cluster,Thomas
4,[Homework Version] Article Reviewer/Fact Check...,Working List,5,Assignee,Thomas
...,...,...,...,...,...
84,[Homework Version] Xometry SEO Article Trackin...,Current Koray Content Workstream,23,Days Since Writing Publish,Xometry
85,[Homework Version] Xometry SEO Article Trackin...,Current Koray Content Workstream,24,Refresh Status,Xometry
86,[Homework Version] Xometry SEO Article Trackin...,Current Koray Content Workstream,25,Main Koray Tracking Sheet Name,Xometry
87,[Homework Version] Xometry SEO Article Trackin...,Current Koray Content Workstream,26,Jira Sprint,Xometry


The dashboard needs to summarize publishing activity by brand, but the source sheets did not contain a physical brand column. Instead of manually adding a brand field to every source sheet, the code infers the brand from the full Google Sheet name.

The function infer_brand(full_sheet_name) takes the full spreadsheet name as input. It converts that name to lowercase using .lower() so the check works regardless of capitalization. Then it checks whether the sheet name contains "thomas" or "xometry".

If the sheet name contains "thomas", the function returns "Thomas". If it contains "xometry", it returns "Xometry". If neither word is found, it returns "Unknown".

This function is then applied to the column audit DataFrame. The result is a new column called inferred_brand. This allows the dashboard to include brand-level summaries without requiring every source sheet to manually include a brand column.

##Step 5: Load Source Sheet Data Into DataFrames

In [ ]:
# ==========================================
# 5. LOAD SOURCE SHEET DATA INTO DATAFRAMES
# ==========================================

all_source_data = []

for source in source_sheets:
    sheet_id = source["sheet_id"]
    tab_name = source["tab_name"]

    spreadsheet = gc.open_by_key(sheet_id)
    worksheet = spreadsheet.worksheet(tab_name)

    values = worksheet.get_all_values()

    if len(values) == 0:
        print(f"{tab_name} is empty and will be skipped.")
        continue

    header_row = values[0]
    data_rows = values[1:]

    df = pd.DataFrame(data_rows, columns=header_row)

    full_sheet_name = spreadsheet.title
    brand = infer_brand(full_sheet_name)

    df["full_sheet_name"] = full_sheet_name
    df["tab_name"] = tab_name
    df["inferred_brand"] = brand

    all_source_data.append(df)

    print(f"Loaded {len(df)} rows from {full_sheet_name} - {tab_name}")

Loaded 761 rows from [Homework Version] Article Reviewer/Fact Checker Tracking Sheet (Thomas) - Working List
Loaded 323 rows from [Homework Version] Industry Overview Pages - Thomas Strategy Plan - Topics
Loaded 130 rows from [Homework Version] Xometry SEO Article Tracking Sheet - Current Koray Content Workstream


After auditing the columns, the automation loads the actual source data. The code creates an empty list called all_source_data. It then loops through each source sheet in the source_sheets list.

For each source, the code opens the Google Sheet, selects the correct worksheet tab, and pulls all values. If a tab is empty, the code prints a message and skips that source using continue. This prevents the script from failing if a source tab has no data.

The first row is stored as the header row, and all remaining rows are stored as data rows. The line pd.DataFrame(data_rows, columns=header_row) converts the raw Google Sheets data into a pandas DataFrame.

The code also adds three metadata columns:

full_sheet_name
tab_name
inferred_brand

These metadata fields make the combined data easier to trace back to its source. This is helpful for quality control because if a row looks incorrect later, the user can see which spreadsheet and tab it came from.

Each DataFrame is appended to all_source_data, creating a list of all loaded source tables.

##Step 6: Preview the Loaded Data

In [ ]:
# ==========================================
# 6. PREVIEW LOADED DATA
# ==========================================

for df in all_source_data:
    print("\n==========================================")
    print(f"Source: {df['full_sheet_name'].iloc[0]}")
    print(f"Tab: {df['tab_name'].iloc[0]}")
    print(f"Brand: {df['inferred_brand'].iloc[0]}")
    print(f"Rows: {df.shape[0]}")
    print(f"Columns: {df.shape[1]}")
    display(df.head())


Source: [Homework Version] Article Reviewer/Fact Checker Tracking Sheet (Thomas)
Tab: Working List
Brand: Thomas
Rows: 761
Columns: 40


,article_url,Wordpress Link,Title,topic_cluster,Assignee,status,Google Doc Link,Infographic Requested,Infographic Status,Engineer Review Images,...,Writing Latest Activity,Images Latest Activity,Days Since Publish,Refresh Status,Designer,Jira Sprint,Article ID,full_sheet_name,tab_name,inferred_brand
0,https://www.thomasnet.com/articles/machinery-t...,https://thomasmkt.wpengine.com/wp-admin/post.p...,,machinery tools supplies,Mahder T.,Published,WordPress Site,,Published,TRUE,...,2024-11-18,,533,🛑 Refresh Overdue,,Sprint 2024-11-06 to 2024-11-20,THOM-001,[Homework Version] Article Reviewer/Fact Check...,Working List,Thomas
1,https://www.thomasnet.com/articles/procurement...,https://thomasmkt.wpengine.com/wp-admin/post.p...,,procurement,Mahder T.,Published,Review completed on WordPress Site,TRUE,Published,TRUE,...,2024-11-25,,526,🛑 Refresh Overdue,,Sprint 2024-11-20 to 2024-12-04,THOM-002,[Homework Version] Article Reviewer/Fact Check...,Working List,Thomas
2,https://www.thomasnet.com/articles/plant-facil...,https://thomasmkt.wpengine.com/wp-admin/post.p...,,plant facility equipment,Mahder T.,Published,WordPress Site,,Published,TRUE,...,2024-11-26,,525,🛑 Refresh Overdue,,Sprint 2024-11-20 to 2024-12-04,THOM-003,[Homework Version] Article Reviewer/Fact Check...,Working List,Thomas
3,https://www.thomasnet.com/articles/plant-facil...,https://thomasmkt.wpengine.com/wp-admin/post.p...,,plant facility equipment,Mahder T.,Published,Review completed on WordPress Site,,Published,TRUE,...,2025-02-06,,453,🛑 Refresh Overdue,,Sprint 2025-01-29 to 2025-02-12,THOM-004,[Homework Version] Article Reviewer/Fact Check...,Working List,Thomas
4,https://www.thomasnet.com/articles/other/types...,https://thomasmkt.wpengine.com/wp-admin/post.p...,,other,Hani G.,Published,Review completed on WordPress Site,,Published,TRUE,...,2024-11-26,,525,🛑 Refresh Overdue,,Sprint 2024-11-20 to 2024-12-04,THOM-005,[Homework Version] Article Reviewer/Fact Check...,Working List,Thomas



Source: [Homework Version] Industry Overview Pages - Thomas Strategy Plan
Tab: Topics
Brand: Thomas
Rows: 323
Columns: 28


,Heading SRP,status,publish_date,Writer,Deadline,Heading Link,WordPress Article Links,article_url,Types of Companies,Include Canada? Y/N,...,Notes,Supplier Quote,Heading ID,Refresh Date Pull,Refresh Date Published,Jira Sprint,topic_cluster,full_sheet_name,tab_name,inferred_brand
0,Plastic Injection Molding,Published,2025-02-03,Rebecca,2025-02-03,https://www.thomasnet.com/suppliers/usa/inject...,https://thomasmkt.wpengine.com/wp-admin/post.p...,https://www.thomasnet.com/articles/top-supplie...,,Yes,...,Map w/o Canada https://app.air.inc/a/cQthYUBSM,,60310604,,,Sprint 2025-01-29 to 2025-02-12,top suppliers,[Homework Version] Industry Overview Pages - T...,Topics,Thomas
1,Steel,Published,2025-02-11,Rebecca,2025-02-11,https://www.thomasnet.com/suppliers/usa/steel-...,https://thomasmkt.wpengine.com/wp-admin/post.p...,https://www.thomasnet.com/articles/top-supplie...,,No,...,,High priority,79740205,2026-04-06,2026-04-06,Sprint 2025-01-29 to 2025-02-12,top suppliers,[Homework Version] Industry Overview Pages - T...,Topics,Thomas
2,Automotive Parts,Published,2025-02-24,Rebecca,2025-02-24,https://www.thomasnet.com/suppliers/usa/automo...,https://thomasmkt.wpengine.com/wp-admin/post.p...,https://www.thomasnet.com/articles/top-supplie...,,No,...,,,2311405,2026-04-06,2026-04-06,Sprint 2025-02-12 to 2025-02-26,top suppliers,[Homework Version] Industry Overview Pages - T...,Topics,Thomas
3,Aluminum,Published,2025-03-02,Rebecca,2025-03-02,https://www.thomasnet.com/suppliers/search?cov...,https://thomasmkt.wpengine.com/wp-admin/post.p...,https://www.thomasnet.com/articles/top-supplie...,,No,...,,High priority,1261809,2026-04-06,2026-04-06,Sprint 2025-02-26 to 2025-03-12,top suppliers,[Homework Version] Industry Overview Pages - T...,Topics,Thomas
4,Cranes,Published,2025-03-06,Kat,3/4/2025,https://www.thomasnet.com/suppliers/usa/cranes...,https://thomasmkt.wpengine.com/wp-admin/post.p...,https://www.thomasnet.com/articles/top-supplie...,,No,...,,Published,20690806,,,Sprint 2025-02-26 to 2025-03-12,top suppliers,[Homework Version] Industry Overview Pages - T...,Topics,Thomas



Source: [Homework Version] Xometry SEO Article Tracking Sheet
Tab: Current Koray Content Workstream
Brand: Xometry
Rows: 130
Columns: 30


,Topic,article_url,Prismic URL,Target Keyword (slug),topic_cluster,Content Type,Koray Content Brief,Article Google Doc,status,publish_date,...,Quotes Latest Activity,Images Latest Activity,Days Since Writing Publish,Refresh Status,Main Koray Tracking Sheet Name,Jira Sprint,Article ID,full_sheet_name,tab_name,inferred_brand
0,Plastic Injection Molding,https://www.xometry.com/resources/injection-mo...,https://xometry-marketing.prismic.io/builder/p...,basics of plastic injection molding,injection molding,Existing Node,https://docs.google.com/spreadsheets/d/1ZSO_Cw...,https://docs.google.com/document/d/1mRqO9FFfsA...,Published,2025-12-08,...,2025-12-09,2026-04-21,148,,Injection Molding (Xometry - 3),Sprint 2025-12-03 to 2025-12-17,KOR-001,[Homework Version] Xometry SEO Article Trackin...,Current Koray Content Workstream,Xometry
1,"Injection Molding: Definition, Types, Diagram",https://www.xometry.com/resources/injection-mo...,https://xometry-marketing.prismic.io/builder/p...,types of injection molding technology,injection molding,Existing Node,https://docs.google.com/spreadsheets/d/1ZSO_Cw...,https://docs.google.com/document/d/1l_jeDrllav...,Published,2025-12-08,...,2025-12-09,2026-03-18,148,,Injection Molding (Xometry - 3),Sprint 2025-12-03 to 2025-12-17,KOR-002,[Homework Version] Xometry SEO Article Trackin...,Current Koray Content Workstream,Xometry
2,Parts of an Injection Molding Machine,https://www.xometry.com/resources/injection-mo...,https://xometry-marketing.prismic.io/builder/p...,injection mold components,injection molding,Existing Node,https://docs.google.com/spreadsheets/d/1ZSO_Cw...,https://docs.google.com/document/d/1h6kYPJDVxa...,Published,2025-12-08,...,2025-12-09,2026-02-23,148,,Injection Molding (Xometry - 3),Sprint 2025-12-03 to 2025-12-17,KOR-003,[Homework Version] Xometry SEO Article Trackin...,Current Koray Content Workstream,Xometry
3,Thin-Wall Injection Molding,https://www.xometry.com/resources/injection-mo...,https://xometry-marketing.prismic.io/builder/p...,thin wall injection molding,injection molding,New Node,https://docs.google.com/spreadsheets/d/1ZSO_Cw...,https://docs.google.com/document/d/1MAOhh1xXCv...,Published,2025-12-08,...,2025-12-09,2026-02-23,148,,Injection Molding (Xometry - 3),Sprint 2025-12-03 to 2025-12-17,KOR-005,[Homework Version] Xometry SEO Article Trackin...,Current Koray Content Workstream,Xometry
4,Polypropylene (PP),https://www.xometry.com/resources/materials/po...,https://xometry-marketing.prismic.io/builder/p...,polypropylene,materials,Existing Node,https://docs.google.com/spreadsheets/d/1ZSO_Cw...,https://docs.google.com/document/d/1bzhe763Hns...,Published,2025-12-08,...,2025-12-09,2026-02-23,148,,Injection Molding (Xometry - 3),Sprint 2025-12-03 to 2025-12-17,KOR-006,[Homework Version] Xometry SEO Article Trackin...,Current Koray Content Workstream,Xometry


The preview step displays basic information about each loaded DataFrame. The code loops through all_source_data and prints the source name, tab name, inferred brand, row count, and column count.

The line display(df.head()) shows the first five rows of each DataFrame. This is important because it allows the user to visually confirm that the data was loaded correctly before any transformations are applied.

This step supports transparency. Instead of immediately filtering the data, the notebook shows what Python received from the source sheets.

##Step 7: Standardize Column Names

In [ ]:
# ==========================================
# 7. STANDARDIZE COLUMN NAMES
# ==========================================

def clean_column_name(column_name):
    column_name = str(column_name)
    column_name = column_name.strip()
    column_name = column_name.lower()
    column_name = column_name.replace(" ", "_")
    column_name = column_name.replace("-", "_")
    column_name = column_name.replace("/", "_")
    return column_name


standardized_data = []

for df in all_source_data:
    df_copy = df.copy()

    original_columns = list(df_copy.columns)
    cleaned_columns = [clean_column_name(col) for col in original_columns]

    df_copy.columns = cleaned_columns

    standardized_data.append(df_copy)

    print("\nColumns standardized for:")
    print(f"Source: {df_copy['full_sheet_name'].iloc[0]}")
    print(f"Tab: {df_copy['tab_name'].iloc[0]}")
    print("Original columns:")
    print(original_columns)
    print("Cleaned columns:")
    print(cleaned_columns)


Columns standardized for:
Source: [Homework Version] Article Reviewer/Fact Checker Tracking Sheet (Thomas)
Tab: Working List
Original columns:
['article_url', 'Wordpress Link', 'Title', 'topic_cluster', 'Assignee', 'status', 'Google Doc Link', 'Infographic Requested', 'Infographic Status', 'Engineer Review Images', 'Creative Brief', 'Engineer Review Brief', 'Freelancer Creative Brief', 'Notes', 'Final Graphic Link', 'publish_date', 'Infographic Date Published', 'Batch', 'Workflows Complete', 'Live Status', 'Infographic Brief Google Doc Title', 'Koray Content Brief', 'Internal Linking Complete', 'Article Google Doc Title', 'Wordpress Link (for internal links)', 'Published URL (for internal links)', 'Publish Date (for internal links)', 'Status (For internal links)', 'Emma Added Link', 'Total Images', 'Writing Latest Activity', 'Images Latest Activity', 'Days Since Publish', 'Refresh Status', 'Designer', 'Jira Sprint', 'Article ID', 'full_sheet_name', 'tab_name', 'inferred_brand']
Cleane

The source sheets were updated so the important fields use standard names, but the script still includes a cleaning function to make column names more reliable.

The function clean_column_name(column_name) converts each column name to a string, removes extra spaces, changes the text to lowercase, and replaces spaces, hyphens, and slashes with underscores. For example, a column called Publish Date would become publish_date.

The code then loops through each DataFrame, creates a copy, stores the original column names, applies the cleaning function to each column, and replaces the original headers with the cleaned headers.

This step matters because Python treats small differences in column names as different fields. For example, Status, status, and status would be read as different column names unless they are standardized. Cleaning the column names reduces the risk of errors later in the workflow.

##Step 8: Validate Required Columns

In [ ]:
# ==========================================
# 8. VALIDATE REQUIRED COLUMNS
# ==========================================

required_source_columns = [
    "article_url",
    "topic_cluster",
    "publish_date",
    "status"
]

final_dashboard_columns = [
    "article_url",
    "topic_cluster",
    "publish_date",
    "status",
    "brand",
    "full_sheet_name",
    "tab_name"
]

validated_data = []

for df in standardized_data:
    source_name = df["full_sheet_name"].iloc[0]
    tab_name = df["tab_name"].iloc[0]

    missing_columns = []

    for column in required_source_columns:
        if column not in df.columns:
            missing_columns.append(column)

    print("\n==========================================")
    print(f"Source: {source_name}")
    print(f"Tab: {tab_name}")

    if missing_columns:
        print(f"Missing required columns: {missing_columns}")
        print("This sheet will be skipped until the missing columns are fixed.")
    else:
        print("All required source columns are present.")

        dashboard_df = df[
            required_source_columns + ["full_sheet_name", "tab_name", "inferred_brand"]
        ].copy()

        dashboard_df = dashboard_df.rename(columns={"inferred_brand": "brand"})

        dashboard_df = dashboard_df[final_dashboard_columns]

        validated_data.append(dashboard_df)


Source: [Homework Version] Article Reviewer/Fact Checker Tracking Sheet (Thomas)
Tab: Working List
All required source columns are present.

Source: [Homework Version] Industry Overview Pages - Thomas Strategy Plan
Tab: Topics
All required source columns are present.

Source: [Homework Version] Xometry SEO Article Tracking Sheet
Tab: Current Koray Content Workstream
All required source columns are present.


This step checks whether each source sheet contains the required fields for the dashboard.

The list required_source_columns includes:

article_url
topic_cluster
publish_date
status

These are the fields that must exist in the source sheets. They are required because the dashboard logic needs to know whether an article has a URL, which topic cluster it belongs to, when it was published, and whether its status is published.

The list final_dashboard_columns includes the source columns plus:

brand
full_sheet_name
tab_name

The brand field is created by Python from inferred_brand, while full_sheet_name and tab_name are used for traceability.

The code loops through each standardized DataFrame and checks whether any required columns are missing. If required fields are missing, that source sheet is skipped and a message is printed. If all required columns are present, the code creates a smaller dashboard-ready DataFrame containing only the columns needed for the report.

The line dashboard_df = dashboard_df.rename(columns={"inferred_brand": "brand"}) changes the Python-created inferred_brand field into the final brand column used in the dashboard.

This validation step is important because it prevents the automation from producing a misleading report if a source sheet is missing a required field.

##Step 9: Combine the Validated Source Data

In [ ]:
# ==========================================
# 9. COMBINE VALIDATED SOURCE DATA
# ==========================================

if len(validated_data) == 0:
    raise ValueError("No valid source data was found. Please check the required columns in the source sheets.")

combined_df = pd.concat(validated_data, ignore_index=True)

print("Validated source data combined successfully.")
print(f"Total rows in combined dataset: {combined_df.shape[0]}")
print(f"Total columns in combined dataset: {combined_df.shape[1]}")

display(combined_df.head())

Validated source data combined successfully.
Total rows in combined dataset: 1214
Total columns in combined dataset: 7


,article_url,topic_cluster,publish_date,status,brand,full_sheet_name,tab_name
0,https://www.thomasnet.com/articles/machinery-t...,machinery tools supplies,2024-11-18,Published,Thomas,[Homework Version] Article Reviewer/Fact Check...,Working List
1,https://www.thomasnet.com/articles/procurement...,procurement,2024-11-25,Published,Thomas,[Homework Version] Article Reviewer/Fact Check...,Working List
2,https://www.thomasnet.com/articles/plant-facil...,plant facility equipment,2024-11-26,Published,Thomas,[Homework Version] Article Reviewer/Fact Check...,Working List
3,https://www.thomasnet.com/articles/plant-facil...,plant facility equipment,2025-02-06,Published,Thomas,[Homework Version] Article Reviewer/Fact Check...,Working List
4,https://www.thomasnet.com/articles/other/types...,other,2024-11-26,Published,Thomas,[Homework Version] Article Reviewer/Fact Check...,Working List


Once each source sheet has passed validation, the automation combines the usable data into one dataset.

The code first checks whether validated_data is empty. If no valid source data exists, the script raises a ValueError with a clear message. This is better than allowing the script to fail later with a confusing error.

The code then creates clean_validated_data. For each validated DataFrame, it removes duplicate columns and resets the row index. This cleanup was added because duplicate column names can cause pandas to throw an InvalidIndexError when combining DataFrames.

The line pd.concat(clean_validated_data, ignore_index=True) stacks all validated DataFrames into one combined DataFrame called combined_df. The argument ignore_index=True resets the row numbers so the combined dataset has one clean index.

This step replaces the manual work of copying rows from multiple trackers into one reporting dataset.

##Step 10: Clean and Filter Month-to-Date Published Articles

In [ ]:
# ==========================================
# 10. CLEAN AND FILTER MONTH-TO-DATE PUBLISHED ARTICLES
# ==========================================

from datetime import datetime

cleaned_df = combined_df.copy()

cleaned_df["article_url"] = cleaned_df["article_url"].astype(str).str.strip()
cleaned_df["topic_cluster"] = cleaned_df["topic_cluster"].astype(str).str.strip()
cleaned_df["status"] = cleaned_df["status"].astype(str).str.strip()
cleaned_df["brand"] = cleaned_df["brand"].astype(str).str.strip()

cleaned_df["publish_date"] = pd.to_datetime(
    cleaned_df["publish_date"],
    errors="coerce"
)

current_date = datetime.now()
current_year = current_date.year
current_month = current_date.month

mtd_published_df = cleaned_df[
    (cleaned_df["article_url"] != "") &
    (cleaned_df["article_url"].str.lower() != "nan") &
    (cleaned_df["status"].str.lower() == "published") &
    (cleaned_df["publish_date"].dt.year == current_year) &
    (cleaned_df["publish_date"].dt.month == current_month)
].copy()

print("Month-to-date published dataset created.")
print(f"Total qualifying rows: {mtd_published_df.shape[0]}")

display(mtd_published_df.head())

Month-to-date published dataset created.
Total qualifying rows: 35


,article_url,topic_cluster,publish_date,status,brand,full_sheet_name,tab_name
100,https://www.thomasnet.com/articles/instruments...,instruments controls,2026-05-01,Published,Thomas,[Homework Version] Article Reviewer/Fact Check...,Working List
101,https://www.thomasnet.com/articles/plant-facil...,plant facility equipment,2026-05-01,Published,Thomas,[Homework Version] Article Reviewer/Fact Check...,Working List
122,https://www.thomasnet.com/articles/engineering...,engineering consulting,2026-05-01,Published,Thomas,[Homework Version] Article Reviewer/Fact Check...,Working List
123,https://www.thomasnet.com/articles/custom-manu...,custom manufacturing fabricating,2026-05-01,Published,Thomas,[Homework Version] Article Reviewer/Fact Check...,Working List
124,https://www.thomasnet.com/articles/electrical-...,electrical power generation,2026-05-01,Published,Thomas,[Homework Version] Article Reviewer/Fact Check...,Working List


This step applies the main business rule for the dashboard.

The code creates a copy of the combined data called cleaned_df. It then cleans the text fields by converting them to strings and stripping extra spaces. This is done for fields such as article_url, topic_cluster, status, and brand.

The publish date column is converted into a real date using pd.to_datetime(). This is necessary because spreadsheet dates may come into Python as text. The argument errors="coerce" tells pandas to convert invalid or blank dates into missing date values instead of crashing.

The code then identifies the current reporting year and month. In production, this allows the dashboard to update automatically based on the current month. For the class demonstration, mock May records were added so the dashboard could show output for May 2026.

The filtered DataFrame mtd_published_df keeps only rows where:

article_url is not blank
status equals Published
publish_date is in the current reporting year
publish_date is in the current reporting month

This is the core decision rule of the automation. It replaces the manual process of checking whether each row should be included in the weekly month-to-date dashboard.

##Step 11: Create Dashboard Summary Tables

In [ ]:
# ==========================================
# 11. CREATE DASHBOARD SUMMARY TABLES
# ==========================================

dashboard_data = mtd_published_df.copy()

dashboard_data["topic_cluster"] = dashboard_data["topic_cluster"].replace("", "Unknown")
dashboard_data["brand"] = dashboard_data["brand"].replace("", "Unknown")

dashboard_data["publish_week"] = (
    dashboard_data["publish_date"]
    .dt.to_period("W")
    .astype(str)
)

brand_scorecards = (
    dashboard_data
    .groupby("brand")
    .agg(
        total_published_articles=("article_url", "count"),
        number_of_topic_clusters=("topic_cluster", "nunique"),
        first_publish_date=("publish_date", "min"),
        latest_publish_date=("publish_date", "max")
    )
    .reset_index()
)

top_topic_by_brand = (
    dashboard_data
    .groupby(["brand", "topic_cluster"])
    .size()
    .reset_index(name="article_count")
    .sort_values(["brand", "article_count"], ascending=[True, False])
    .drop_duplicates(subset="brand")
    .rename(columns={"topic_cluster": "top_topic_cluster"})
)

brand_scorecards = brand_scorecards.merge(
    top_topic_by_brand[["brand", "top_topic_cluster"]],
    on="brand",
    how="left"
)

article_share_by_brand = (
    dashboard_data
    .groupby("brand")
    .size()
    .reset_index(name="article_count")
)

total_articles = article_share_by_brand["article_count"].sum()

article_share_by_brand["article_share_percent"] = (
    article_share_by_brand["article_count"] / total_articles * 100
).round(2)

topic_cluster_by_brand = (
    dashboard_data
    .groupby(["brand", "topic_cluster"])
    .size()
    .reset_index(name="article_count")
    .sort_values(["brand", "article_count"], ascending=[True, False])
)

overall_topic_cluster_summary = (
    dashboard_data
    .groupby("topic_cluster")
    .size()
    .reset_index(name="article_count")
    .sort_values("article_count", ascending=False)
)

weekly_publishing_pace = (
    dashboard_data
    .groupby("publish_week")
    .size()
    .reset_index(name="article_count")
    .sort_values("publish_week")
)

print("Dashboard summary tables created.")

print("\nBrand Scorecards")
display(brand_scorecards)

print("\nArticle Share by Brand")
display(article_share_by_brand)

print("\nTopic Cluster by Brand")
display(topic_cluster_by_brand)

print("\nOverall Topic Cluster Summary")
display(overall_topic_cluster_summary)

print("\nWeekly Publishing Pace")
display(weekly_publishing_pace)

Dashboard summary tables created.

Brand Scorecards


,brand,total_published_articles,number_of_topic_clusters,first_publish_date,latest_publish_date,top_topic_cluster
0,Thomas,20,7,2026-05-01,2026-05-05,top suppliers
1,Xometry,15,3,2026-05-01,2026-05-05,machining



Article Share by Brand


,brand,article_count,article_share_percent
0,Thomas,20,57.14
1,Xometry,15,42.86



Topic Cluster by Brand


,brand,topic_cluster,article_count
6,Thomas,top suppliers,8
1,Thomas,electrical power generation,4
5,Thomas,pumps valves accessories,3
2,Thomas,engineering consulting,2
0,Thomas,custom manufacturing fabricating,1
3,Thomas,instruments controls,1
4,Thomas,plant facility equipment,1
8,Xometry,machining,11
9,Xometry,materials,3
7,Xometry,injection molding,1



Overall Topic Cluster Summary


,topic_cluster,article_count
5,machining,11
9,top suppliers,8
1,electrical power generation,4
6,materials,3
8,pumps valves accessories,3
2,engineering consulting,2
0,custom manufacturing fabricating,1
3,injection molding,1
4,instruments controls,1
7,plant facility equipment,1



Weekly Publishing Pace


,publish_week,article_count
0,2026-04-27/2026-05-03,30
1,2026-05-04/2026-05-10,5


After the month-to-date data is filtered, the automation creates the summary tables needed for the dashboard.

The code starts with dashboard_data = mtd_published_df.copy(). This creates a separate working DataFrame for dashboard calculations.

The code creates a publish_week field by converting each publish date into a weekly period. This allows the dashboard to show publishing pace by week.

The brand scorecard table is created with .groupby("brand").agg(...). This groups the data by brand and calculates:

total published articles
number of unique topic clusters
first publish date
latest publish date

The code also creates a top_topic_by_brand table. It groups the data by brand and topic cluster, counts the articles in each combination, sorts the results, and keeps the top topic cluster for each brand.

The article share table calculates how many articles came from each brand and what percentage of total month-to-date output each brand represents.

The topic cluster tables summarize activity by topic cluster, both overall and by brand. The weekly publishing pace table groups the data by publish week and counts how many articles were published in each week.

These summary tables are important because they turn row-level article data into business metrics.

##Step 12: Create and Export a Filterable Interactive HTML Dashboard

In [ ]:
# ==========================================
# 12. CREATE THE FINAL DASHBOARD
# ==========================================

import json
import pandas as pd
from google.colab import files
from IPython.display import HTML, display

# Create a copy of the final month-to-date dataset for the dashboard.
interactive_data = mtd_published_df.copy()

# Clean text fields so labels appear consistently in the dashboard.
interactive_data["topic_cluster"] = (
    interactive_data["topic_cluster"]
    .astype(str)
    .str.strip()
    .replace({"": "Unknown", "nan": "Unknown"})
    .str.replace("_", " ", regex=False)
    .str.title()
)

interactive_data["brand"] = (
    interactive_data["brand"]
    .astype(str)
    .str.strip()
    .replace({"": "Unknown", "nan": "Unknown"})
    .str.title()
)

# Make sure publish_date is a real date.
interactive_data["publish_date"] = pd.to_datetime(
    interactive_data["publish_date"],
    errors="coerce"
)

# Remove rows where the publish date could not be converted.
interactive_data = interactive_data.dropna(subset=["publish_date"]).copy()

# Create the reporting month label from the month used in the earlier filter step.
try:
    reporting_month_start = pd.Timestamp(year=current_year, month=current_month, day=1)
except NameError:
    reporting_month_start = pd.Timestamp.today().replace(day=1)

report_month_label = reporting_month_start.strftime("%B %Y")
report_generated_label = pd.Timestamp.now(tz='America/New_York').strftime("%B %d, %Y")

# Convert dates to text so they can be safely used inside the HTML dashboard.
interactive_data["publish_date"] = interactive_data["publish_date"].dt.strftime("%Y-%m-%d")

# Keep only the columns needed inside the interactive dashboard.
dashboard_records = interactive_data[
    [
        "article_url",
        "topic_cluster",
        "publish_date",
        "brand",
        "full_sheet_name",
        "tab_name"
    ]
].to_dict(orient="records")

# Convert the dashboard data into JSON so JavaScript can read it.
records_json = json.dumps(dashboard_records)

# ==========================================
# 12A. GENERATE EXECUTIVE SUMMARY
# ==========================================

total_published = len(interactive_data)

thomas_total = interactive_data[interactive_data["brand"] == "Thomas"].shape[0]
xometry_total = interactive_data[interactive_data["brand"] == "Xometry"].shape[0]

topic_counts = (
    interactive_data
    .groupby("topic_cluster")
    .size()
    .reset_index(name="article_count")
    .sort_values("article_count", ascending=False)
)

weekly_counts = (
    interactive_data
    .assign(
        publish_week_label=interactive_data["publish_date"].apply(
            lambda date_value: "Week " + str(((pd.to_datetime(date_value).day - 1) // 7) + 1)
        )
    )
    .groupby("publish_week_label")
    .size()
    .reset_index(name="article_count")
    .sort_values("publish_week_label")
)

if len(topic_counts) > 0:
    top_topic_cluster = topic_counts.iloc[0]["topic_cluster"]
else:
    top_topic_cluster = "No Data"

summary_prompt = f"""
Write a concise executive summary for a weekly publishing pace dashboard.

Reporting month: {report_month_label}

Use these dashboard metrics:
- Total published month-to-date: {total_published}
- Thomas published month-to-date: {thomas_total}
- Xometry published month-to-date: {xometry_total}
- Top topic cluster: {top_topic_cluster}

Weekly publishing pace:
{weekly_counts.to_string(index=False)}

Topic cluster summary:
{topic_counts.to_string(index=False)}

Write 3 to 4 business-friendly sentences.
Mention publishing pace, brand contribution, and topic cluster concentration.
Do not overstate conclusions.
Do not use bullet points.
"""

try:
    import os
    from google.colab import userdata
    from openai import OpenAI

    # Pull the OpenAI API key from Colab Secrets.
    # The secret must be named exactly: openaikey
    os.environ["OPENAI_API_KEY"] = userdata.get("openaikey")

    client = OpenAI()

    summary_response = client.responses.create(
        model="gpt-4.1-mini",
        instructions="You write concise business summaries for analytics dashboards.",
        input=summary_prompt
    )

    executive_summary = summary_response.output_text.strip()

except Exception as error:
    executive_summary = (
        f"For {report_month_label}, the dashboard summarizes month-to-date publishing "
        f"performance across brands, topic clusters, and publishing weeks. The report includes "
        f"{total_published} published article records, with {thomas_total} from Thomas and "
        f"{xometry_total} from Xometry. The leading topic cluster is {top_topic_cluster}. "
        "The AI-generated executive summary could not be created automatically, but the dashboard "
        "tables, scorecards, filters, and visuals were still generated successfully."
    )

# Convert dashboard text into JSON-safe values so JavaScript can read them.
executive_summary_json = json.dumps(executive_summary)
report_month_json = json.dumps(report_month_label)
report_generated_json = json.dumps(report_generated_label)

# ==========================================
# 12B. BUILD THE INTERACTIVE HTML DASHBOARD
# ==========================================

html_template = """
<!DOCTYPE html>
<html>
<head>
    <meta charset=\"UTF-8\">
    <title>Weekly Publishing Pace Dashboard</title>
    <script src=\"https://cdn.plot.ly/plotly-2.35.2.min.js\"></script>

    <style>
        body {
            font-family: Arial, sans-serif;
            background-color: #f6f8fb;
            color: #1f2d3d;
            margin: 0;
            padding: 24px;
        }

        .dashboard-container {
            max-width: 1450px;
            margin: 0 auto;
            background-color: #ffffff;
            border: 1px solid #d9e2ec;
            border-radius: 14px;
            padding: 26px;
            box-shadow: 0 4px 14px rgba(0, 0, 0, 0.06);
        }

        .dashboard-title {
            text-align: center;
            margin-bottom: 4px;
            font-size: 28px;
            font-weight: 700;
            color: #12355b;
        }

        .dashboard-subtitle {
            text-align: center;
            margin-bottom: 12px;
            font-size: 15px;
            color: #526d82;
        }

        .report-meta-row {
            display: flex;
            justify-content: center;
            gap: 12px;
            flex-wrap: wrap;
            margin-bottom: 22px;
        }

        .report-chip {
            background-color: #eef4fb;
            border: 1px solid #d9e7f5;
            color: #12355b;
            border-radius: 999px;
            padding: 8px 14px;
            font-size: 13px;
            font-weight: 700;
        }

        .filter-panel {
            display: flex;
            flex-wrap: wrap;
            gap: 14px;
            align-items: end;
            background-color: #eef4fb;
            border: 1px solid #d9e7f5;
            border-radius: 12px;
            padding: 16px;
            margin-bottom: 14px;
        }

        .filter-block {
            display: flex;
            flex-direction: column;
            gap: 5px;
        }

        .filter-block label {
            font-size: 12px;
            font-weight: 700;
            color: #12355b;
        }

        select, button {
            min-width: 190px;
            padding: 9px 10px;
            border-radius: 8px;
            border: 1px solid #b8c7d9;
            background-color: #ffffff;
            color: #1f2d3d;
            font-size: 14px;
        }

        button {
            min-width: 120px;
            cursor: pointer;
            background-color: #12355b;
            color: #ffffff;
            border: none;
            font-weight: 700;
        }

        .filter-status {
            flex-basis: 100%;
            font-size: 13px;
            color: #526d82;
            margin-top: 4px;
        }

        .executive-summary {
            background-color: #f7fbff;
            border: 1px solid #d9e7f5;
            border-left: 5px solid #1f5f99;
            border-radius: 12px;
            padding: 16px 18px;
            margin-bottom: 22px;
        }

        .executive-summary-title {
            font-size: 15px;
            font-weight: 800;
            color: #12355b;
            margin-bottom: 8px;
        }

        .executive-summary-text {
            font-size: 14px;
            line-height: 1.55;
            color: #334e68;
        }

        .scorecard-grid {
            display: grid;
            grid-template-columns: repeat(4, 1fr);
            gap: 14px;
            margin-bottom: 22px;
        }

        .scorecard {
            border: 1px solid #d9e2ec;
            border-radius: 12px;
            padding: 18px 12px;
            text-align: center;
            background-color: #fbfdff;
        }

        .scorecard-title {
            font-size: 13px;
            font-weight: 700;
            color: #334e68;
            margin-bottom: 10px;
        }

        .scorecard-value {
            font-size: 38px;
            font-weight: 800;
            color: #1f5f99;
        }

        .scorecard-small {
            font-size: 18px;
            font-weight: 800;
            color: #1f5f99;
            line-height: 1.25;
            min-height: 46px;
            display: flex;
            align-items: center;
            justify-content: center;
        }

        .chart-grid {
            display: grid;
            grid-template-columns: 1fr 1fr 1.35fr;
            gap: 18px;
            margin-bottom: 18px;
        }

        .chart-grid-bottom {
            display: grid;
            grid-template-columns: 1fr 1fr;
            gap: 18px;
        }

        .chart-card {
            border: 1px solid #d9e2ec;
            border-radius: 12px;
            padding: 12px;
            background-color: #ffffff;
            min-height: 390px;
        }

        .chart-card-wide {
            min-height: 390px;
        }

        .footer-note {
            margin-top: 18px;
            padding: 12px;
            border-radius: 10px;
            background-color: #eef4fb;
            color: #526d82;
            text-align: center;
            font-size: 13px;
        }

        @media (max-width: 1000px) {
            .scorecard-grid,
            .chart-grid,
            .chart-grid-bottom {
                grid-template-columns: 1fr;
            }
        }
    </style>
</head>

<body>
    <div class="dashboard-container">
        <div class="dashboard-title">Weekly Publishing Pace Dashboard</div>
        <div class="dashboard-subtitle">Month-to-Date Publishing Progress</div>

        <div class="report-meta-row">
            <div class="report-chip">Reporting Month: <span id="reportMonth"></span></div>
            <div class="report-chip">Generated: <span id="reportGenerated"></span></div>
        </div>

        <div class="filter-panel">
            <div class="filter-block">
                <label for="brandFilter">Filter by Brand</label>
                <select id="brandFilter"></select>
            </div>

            <div class="filter-block">
                <label for="topicFilter">Filter by Topic Cluster</label>
                <select id="topicFilter"></select>
            </div>

            <div class="filter-block">
                <button onclick="clearFilters()">Clear Filters</button>
            </div>

            <div class="filter-status" id="filterStatus"></div>
        </div>

        <div class="executive-summary">
            <div class="executive-summary-title">Executive Summary</div>
            <div class="executive-summary-text" id="executiveSummary"></div>
        </div>

        <div class="scorecard-grid">
            <div class="scorecard">
                <div class="scorecard-title">Thomas Published MTD</div>
                <div class="scorecard-value" id="thomasTotal">0</div>
            </div>

            <div class="scorecard">
                <div class="scorecard-title">Xometry Published MTD</div>
                <div class="scorecard-value" id="xometryTotal">0</div>
            </div>

            <div class="scorecard">
                <div class="scorecard-title">Total Published MTD</div>
                <div class="scorecard-value" id="totalPublished">0</div>
            </div>

            <div class="scorecard">
                <div class="scorecard-title">Top Topic Cluster</div>
                <div class="scorecard-small" id="topTopic">No Data</div>
            </div>
        </div>

        <div class="chart-grid">
            <div class="chart-card">
                <div id="thomasTopicChart"></div>
            </div>

            <div class="chart-card">
                <div id="xometryTopicChart"></div>
            </div>

            <div class="chart-card chart-card-wide">
                <div id="overallTopicChart"></div>
            </div>
        </div>

        <div class="chart-grid-bottom">
            <div class="chart-card">
                <div id="weeklyPaceChart"></div>
            </div>

            <div class="chart-card">
                <div id="brandShareChart"></div>
            </div>
        </div>

        <div class="footer-note">
            This interactive dashboard was generated with Python from Google Sheets source data. Use the filters above to explore the month-to-date publishing data by brand or topic cluster.
        </div>
    </div>

    <script>
        const records = __DASHBOARD_RECORDS__;
        const executiveSummary = __EXECUTIVE_SUMMARY__;
        const reportMonthLabel = __REPORT_MONTH_LABEL__;
        const reportGeneratedLabel = __REPORT_GENERATED_LABEL__;

        const bluePalette = [
            "#0b2545",
            "#133b5c",
            "#1f5f99",
            "#2f80c1",
            "#4b9cd3",
            "#74b3ce",
            "#9ccce3",
            "#c6ddeb",
            "#dbeafe",
            "#edf6fb"
        ];

        const config = {
            responsive: true,
            displaylogo: false
        };

        function uniqueValues(data, key) {
            return [...new Set(data.map(row => row[key]).filter(value => value && value !== "Unknown"))].sort();
        }

        function populateFilters() {
            const brandFilter = document.getElementById("brandFilter");
            const topicFilter = document.getElementById("topicFilter");

            const brands = uniqueValues(records, "brand");
            const topics = uniqueValues(records, "topic_cluster");

            brandFilter.innerHTML = "<option value='All'>All Brands</option>";
            topicFilter.innerHTML = "<option value='All'>All Topic Clusters</option>";

            brands.forEach(brand => {
                brandFilter.innerHTML += `<option value="${brand}">${brand}</option>`;
            });

            topics.forEach(topic => {
                topicFilter.innerHTML += `<option value="${topic}">${topic}</option>`;
            });

            brandFilter.addEventListener("change", updateDashboard);
            topicFilter.addEventListener("change", updateDashboard);
        }

        function clearFilters() {
            document.getElementById("brandFilter").value = "All";
            document.getElementById("topicFilter").value = "All";
            updateDashboard();
        }

        function getFilteredRecords() {
            const selectedBrand = document.getElementById("brandFilter").value;
            const selectedTopic = document.getElementById("topicFilter").value;

            return records.filter(row => {
                const brandMatch = selectedBrand === "All" || row.brand === selectedBrand;
                const topicMatch = selectedTopic === "All" || row.topic_cluster === selectedTopic;
                return brandMatch && topicMatch;
            });
        }

        function groupCount(data, key) {
            const counts = {};

            data.forEach(row => {
                const value = row[key] || "Unknown";
                counts[value] = (counts[value] || 0) + 1;
            });

            return Object.entries(counts)
                .map(([label, count]) => ({ label, count }))
                .sort((a, b) => b.count - a.count);
        }

        function getWeekLabel(dateString) {
            const date = new Date(dateString + "T00:00:00");
            const weekNumber = Math.floor((date.getDate() - 1) / 7) + 1;
            return "Week " + weekNumber;
        }

        function updateScorecards(filteredRecords) {
            const thomasTotal = filteredRecords.filter(row => row.brand === "Thomas").length;
            const xometryTotal = filteredRecords.filter(row => row.brand === "Xometry").length;
            const totalPublished = filteredRecords.length;

            const topicCounts = groupCount(filteredRecords, "topic_cluster");
            const topTopic = topicCounts.length > 0 ? topicCounts[0].label : "No Data";

            document.getElementById("thomasTotal").textContent = thomasTotal;
            document.getElementById("xometryTotal").textContent = xometryTotal;
            document.getElementById("totalPublished").textContent = totalPublished;
            document.getElementById("topTopic").textContent = topTopic;
        }

        function updateFilterStatus(filteredRecords) {
            const selectedBrand = document.getElementById("brandFilter").value;
            const selectedTopic = document.getElementById("topicFilter").value;

            document.getElementById("filterStatus").textContent =
                `Active filters: Brand = ${selectedBrand}; Topic Cluster = ${selectedTopic}. Displaying ${filteredRecords.length} published article records for ${reportMonthLabel}.`;
        }

        function drawDonutChart(containerId, data, title) {
            const grouped = groupCount(data, "topic_cluster");

            if (grouped.length === 0) {
                Plotly.newPlot(
                    containerId,
                    [],
                    {
                        title: { text: title, font: { size: 17, color: "#12355b" } },
                        annotations: [
                            {
                                text: "No data for selected filters",
                                x: 0.5,
                                y: 0.5,
                                showarrow: false,
                                font: { size: 14, color: "#526d82" }
                            }
                        ],
                        paper_bgcolor: "#ffffff",
                        plot_bgcolor: "#ffffff",
                        margin: { l: 20, r: 20, t: 55, b: 20 }
                    },
                    config
                );
                return;
            }

            Plotly.newPlot(
                containerId,
                [
                    {
                        type: "pie",
                        labels: grouped.map(row => row.label),
                        values: grouped.map(row => row.count),
                        hole: 0.42,
                        marker: { colors: bluePalette },
                        textinfo: "percent",
                        textposition: "inside",
                        hovertemplate: "<b>%{label}</b><br>Articles: %{value}<br>Share: %{percent}<extra></extra>",
                        sort: false
                    }
                ],
                {
                    title: { text: title, font: { size: 17, color: "#12355b" } },
                    showlegend: true,
                    legend: {
                        orientation: "h",
                        y: -0.18,
                        x: 0.5,
                        xanchor: "center",
                        font: { size: 10, color: "#334e68" }
                    },
                    paper_bgcolor: "#ffffff",
                    plot_bgcolor: "#ffffff",
                    margin: { l: 15, r: 15, t: 55, b: 85 }
                },
                config
            );
        }

        function drawHorizontalBarChart(containerId, data, title) {
            const grouped = groupCount(data, "topic_cluster")
                .sort((a, b) => a.count - b.count);

            if (grouped.length === 0) {
                Plotly.newPlot(
                    containerId,
                    [],
                    {
                        title: { text: title, font: { size: 17, color: "#12355b" } },
                        annotations: [
                            {
                                text: "No data for selected filters",
                                x: 0.5,
                                y: 0.5,
                                showarrow: false,
                                font: { size: 14, color: "#526d82" }
                            }
                        ],
                        paper_bgcolor: "#ffffff",
                        plot_bgcolor: "#ffffff",
                        margin: { l: 20, r: 20, t: 55, b: 20 }
                    },
                    config
                );
                return;
            }

            Plotly.newPlot(
                containerId,
                [
                    {
                        type: "bar",
                        x: grouped.map(row => row.count),
                        y: grouped.map(row => row.label),
                        orientation: "h",
                        marker: { color: "#2f80c1" },
                        hovertemplate: "<b>%{y}</b><br>Articles: %{x}<extra></extra>"
                    }
                ],
                {
                    title: { text: title, font: { size: 17, color: "#12355b" } },
                    xaxis: {
                        title: "Published Articles",
                        gridcolor: "#d9e7f5",
                        zeroline: false
                    },
                    yaxis: {
                        automargin: true,
                        tickfont: { size: 11 }
                    },
                    paper_bgcolor: "#ffffff",
                    plot_bgcolor: "#f7fbff",
                    margin: { l: 150, r: 20, t: 55, b: 45 }
                },
                config
            );
        }

        function drawWeeklyPaceChart(containerId, data) {
            const weekCounts = {};

            data.forEach(row => {
                const week = getWeekLabel(row.publish_date);
                weekCounts[week] = (weekCounts[week] || 0) + 1;
            });

            const grouped = Object.entries(weekCounts)
                .map(([label, count]) => ({ label, count }))
                .sort((a, b) => Number(a.label.replace("Week ", "")) - Number(b.label.replace("Week ", "")));

            if (grouped.length === 0) {
                Plotly.newPlot(
                    containerId,
                    [],
                    {
                        title: { text: "Weekly Publishing Pace", font: { size: 17, color: "#12355b" } },
                        annotations: [
                            {
                                text: "No data for selected filters",
                                x: 0.5,
                                y: 0.5,
                                showarrow: false,
                                font: { size: 14, color: "#526d82" }
                            }
                        ],
                        paper_bgcolor: "#ffffff",
                        plot_bgcolor: "#ffffff",
                        margin: { l: 20, r: 20, t: 55, b: 20 }
                    },
                    config
                );
                return;
            }

            Plotly.newPlot(
                containerId,
                [
                    {
                        type: "bar",
                        x: grouped.map(row => row.label),
                        y: grouped.map(row => row.count),
                        marker: { color: "#1f5f99" },
                        hovertemplate: "<b>%{x}</b><br>Articles: %{y}<extra></extra>"
                    }
                ],
                {
                    title: { text: "Weekly Publishing Pace", font: { size: 17, color: "#12355b" } },
                    xaxis: {
                        title: "Publish Week"
                    },
                    yaxis: {
                        title: "Published Articles",
                        gridcolor: "#d9e7f5",
                        rangemode: "tozero"
                    },
                    paper_bgcolor: "#ffffff",
                    plot_bgcolor: "#f7fbff",
                    margin: { l: 60, r: 20, t: 55, b: 55 }
                },
                config
            );
        }

        function drawBrandShareChart(containerId, data) {
            const grouped = groupCount(data, "brand");

            if (grouped.length === 0) {
                Plotly.newPlot(
                    containerId,
                    [],
                    {
                        title: { text: "Share of Published Articles by Brand", font: { size: 17, color: "#12355b" } },
                        annotations: [
                            {
                                text: "No data for selected filters",
                                x: 0.5,
                                y: 0.5,
                                showarrow: false,
                                font: { size: 14, color: "#526d82" }
                            }
                        ],
                        paper_bgcolor: "#ffffff",
                        plot_bgcolor: "#ffffff",
                        margin: { l: 20, r: 20, t: 55, b: 20 }
                    },
                    config
                );
                return;
            }

            Plotly.newPlot(
                containerId,
                [
                    {
                        type: "pie",
                        labels: grouped.map(row => row.label),
                        values: grouped.map(row => row.count),
                        hole: 0.55,
                        marker: { colors: ["#1f5f99", "#74b3ce", "#c6ddeb"] },
                        textinfo: "label+percent",
                        textposition: "inside",
                        hovertemplate: "<b>%{label}</b><br>Articles: %{value}<br>Share: %{percent}<extra></extra>"
                    }
                ],
                {
                    title: { text: "Share of Published Articles by Brand", font: { size: 17, color: "#12355b" } },
                    showlegend: false,
                    paper_bgcolor: "#ffffff",
                    plot_bgcolor: "#ffffff",
                    margin: { l: 20, r: 20, t: 55, b: 20 }
                },
                config
            );
        }

        function updateDashboard() {
            const filteredRecords = getFilteredRecords();

            updateScorecards(filteredRecords);
            updateFilterStatus(filteredRecords);

            const thomasRecords = filteredRecords.filter(row => row.brand === "Thomas");
            const xometryRecords = filteredRecords.filter(row => row.brand === "Xometry");

            drawDonutChart("thomasTopicChart", thomasRecords, "Thomas Topic Cluster Breakdown");
            drawDonutChart("xometryTopicChart", xometryRecords, "Xometry Topic Cluster Breakdown");
            drawHorizontalBarChart("overallTopicChart", filteredRecords, "Published Articles by Topic Cluster");
            drawWeeklyPaceChart("weeklyPaceChart", filteredRecords);
            drawBrandShareChart("brandShareChart", filteredRecords);
        }

        document.getElementById("executiveSummary").textContent = executiveSummary;
        document.getElementById("reportMonth").textContent = reportMonthLabel;
        document.getElementById("reportGenerated").textContent = reportGeneratedLabel;

        populateFilters();
        updateDashboard();
    </script>
</body>
</html>
"""

# Insert the Python-generated records, executive summary, and report labels into the HTML template.
html_dashboard = html_template.replace("__DASHBOARD_RECORDS__", records_json)
html_dashboard = html_dashboard.replace("__EXECUTIVE_SUMMARY__", executive_summary_json)
html_dashboard = html_dashboard.replace("__REPORT_MONTH_LABEL__", report_month_json)
html_dashboard = html_dashboard.replace("__REPORT_GENERATED_LABEL__", report_generated_json)

# Save the dashboard as an HTML file.
timestamp = datetime.now().strftime("%Y-%m-%d-%H-%M-%S")
dash_file_name = f"weekly_publishing_pace_dashboard_{timestamp}.html"

with open(dash_file_name, "w", encoding="utf-8") as file:
    file.write(html_dashboard)

# Display the dashboard inside Colab.
display(HTML(html_dashboard))

# Download the dashboard file so it can be opened outside Colab.
files.download(dash_file_name)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

The final dashboard section creates an interactive HTML file instead of a static chart. This makes the output easier for a business user to explore.

The code starts by copying mtd_published_df into interactive_data. It cleans the topic cluster and brand fields so the labels look consistent in the dashboard. It also converts the publish date into a clean date format.

The dashboard records are converted into a list of dictionaries and then into JSON using json.dumps(). This step is necessary because the dashboard uses JavaScript in the HTML file, and JavaScript needs the data in JSON format.

The code also creates the reporting month label, such as May 2026, and a generated date. These are inserted near the top of the dashboard so the user clearly understands the reporting window.

##Conclusion

This automation successfully turns a manual weekly reporting process into a repeatable Python workflow. The script pulls publishing data from multiple Google Sheets, standardizes and validates the required fields, combines the data, filters it to current month-to-date published articles, creates summary tables, and exports an interactive dashboard.
The final dashboard makes it easier to understand publishing performance by brand, topic cluster, and week. It also allows the user to filter the report and review an executive summary without manually rebuilding charts or recalculating totals.


One limitation is that the automation depends on standardized source column names. If a source sheet changes the names of article_url, topic_cluster, publish_date, or status, the validation step will catch the issue, but the source sheet must still be corrected before the dashboard can run. A future version could expand the automation by writing summary tables back to Google Sheets, sending the dashboard by email, or scheduling the script to run automatically each week.

In [ ]:
# ==========================================
# 13. FINAL AUTOMATION SUMMARY
# ==========================================

print("Automation completed successfully.")
print(f"Total source rows reviewed: {combined_df.shape[0]}")
print(f"Total month-to-date published articles included: {mtd_published_df.shape[0]}")
print(f"Dashboard file created: {dash_file_name}")

Automation completed successfully.
Total source rows reviewed: 1214
Total month-to-date published articles included: 35
Dashboard file created: weekly_publishing_pace_dashboard_2026-05-05-05-01-09.html
